# Training Group DRO

This part reconstructs the Group DRO experiment used in the project with `distilbert-base-uncased`, four `identity_any × label` training groups, group-balanced sampling, adaptive Group DRO weights, validation worst-group checkpoint selection, and final test prediction export.

It is designed for Google Colab and saves `results/group_dro/test_predictions.csv`, which is required by the comparison and error-analysis notebooks.

**Important:** if `results/group_dro/best_model/` already contains the previously selected model, you can skip the training section and run directly from **Load Existing Best Model** through **Final Test Evaluation and Export**.


In [ ]:
#from google.colab import drive
#drive.mount("/content/drive")

from pathlib import Path
import os
#PROJECT_ROOT=Path("/content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments")
#if not PROJECT_ROOT.exists(): raise FileNotFoundError(PROJECT_ROOT)
#os.chdir(PROJECT_ROOT)
#print("Project root:",Path.cwd())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments


## Imports and configuration


In [10]:
import json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, WeightedRandomSampler
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm

SEED=42
MODEL_NAME="distilbert-base-uncased"
BATCH_SIZE=16
EVAL_BATCH_SIZE=32
LEARNING_RATE=2e-5
WEIGHT_DECAY=0.01
NUM_EPOCHS=3
WARMUP_RATIO=0.1
DRO_ETA=0.01
NUM_TRAIN_GROUPS=4
OUTPUT_DIR=Path("results/group_dro")
BEST_MODEL_DIR=OUTPUT_DIR/"best_model"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); BEST_MODEL_DIR.mkdir(parents=True,exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


## Load processed datasets


In [11]:
train_dataset=load_from_disk("data/processed/tokenized_train")
val_dataset=load_from_disk("data/processed/tokenized_validation")
test_dataset=load_from_disk("data/processed/tokenized_test")
print("Train:",len(train_dataset)); print("Validation:",len(val_dataset)); print("Test:",len(test_dataset))
IDENTITY_COLUMNS=["male","female","LGBTQ","christian","muslim","other_religions","black","white"]
required={"input_ids","attention_mask","label","identity_any"}|set(IDENTITY_COLUMNS)
for name,d in [("Train",train_dataset),("Validation",val_dataset),("Test",test_dataset)]:
    miss=required-set(d.column_names)
    if miss: raise ValueError(f"{name} missing: {sorted(miss)}")


Train: 267516
Validation: 45047
Test: 132730


## Add (4) Group DRO training groups


In [12]:
GROUP_NAMES={0:"No identity, non-toxic",1:"Identity, non-toxic",2:"No identity, toxic",3:"Identity, toxic"}
def add_training_group(example):
    return {"group_id":int(example["identity_any"])+2*int(example["label"])}
train_dataset=train_dataset.map(add_training_group)
val_dataset=val_dataset.map(add_training_group)
test_dataset=test_dataset.map(add_training_group)
counts=pd.Series(train_dataset["group_id"]).value_counts().sort_index()
display(pd.DataFrame({"group_id":range(4),"group_name":[GROUP_NAMES[i] for i in range(4)],"count":[int(counts.get(i,0)) for i in range(4)]}))


Map:   0%|          | 0/267516 [00:00<?, ? examples/s]

Map:   0%|          | 0/45047 [00:00<?, ? examples/s]

Map:   0%|          | 0/132730 [00:00<?, ? examples/s]

,group_id,group_name,count
0,0,"No identity, non-toxic",142750
1,1,"Identity, non-toxic",94423
2,2,"No identity, toxic",11890
3,3,"Identity, toxic",18453


## Tokenizer, collator, balanced sampler, and loaders


In [13]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
padder=DataCollatorWithPadding(tokenizer=tokenizer,return_tensors="pt")
def collate(features):
    batch=padder([{"input_ids":x["input_ids"],"attention_mask":x["attention_mask"]} for x in features])
    batch["labels"]=torch.tensor([int(x["label"]) for x in features],dtype=torch.long)
    batch["group_id"]=torch.tensor([int(x["group_id"]) for x in features],dtype=torch.long)
    for identity in IDENTITY_COLUMNS:
        batch[identity]=torch.tensor([int(x[identity]) for x in features],dtype=torch.long)
    if "uid" in features[0]: batch["uid"]=torch.tensor([int(x["uid"]) for x in features],dtype=torch.long)
    return batch

gids=np.array(train_dataset["group_id"]); group_counts=np.bincount(gids,minlength=4)
weights=np.array([1.0/group_counts[g] for g in gids])
sampler=WeightedRandomSampler(torch.tensor(weights,dtype=torch.double),num_samples=len(weights),replacement=True)
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,sampler=sampler,collate_fn=collate,pin_memory=torch.cuda.is_available())
val_loader=DataLoader(val_dataset,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate,pin_memory=torch.cuda.is_available())
test_loader=DataLoader(test_dataset,batch_size=EVAL_BATCH_SIZE,shuffle=False,collate_fn=collate,pin_memory=torch.cuda.is_available())
print("Train batches:",len(train_loader),"Validation batches:",len(val_loader),"Test batches:",len(test_loader))


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train batches: 16720 Validation batches: 1408 Test batches: 4148


## Shared group-evaluation function


In [14]:
def official_group_metrics(labels,predictions,identity_data):
    rows=[]; labels=np.asarray(labels); predictions=np.asarray(predictions)
    for identity in IDENTITY_COLUMNS:
        vals=np.asarray(identity_data[identity])
        for y in [0,1]:
            mask=(vals==1)&(labels==y); n=int(mask.sum())
            acc=np.nan if n==0 else accuracy_score(labels[mask],predictions[mask])
            rows.append({"identity":identity,"label":y,"label_name":"non-toxic" if y==0 else "toxic","n":n,"accuracy":acc})
    df=pd.DataFrame(rows); return df,float(df.accuracy.dropna().min())

def evaluate_model(model,data_loader):
    model.eval(); labels_all=[]; preds_all=[]; probs_all=[]; gids_all=[]; ids={i:[] for i in IDENTITY_COLUMNS}; uids=[]; total_loss=0; total_n=0
    loss_fn=nn.CrossEntropyLoss(reduction="none")
    with torch.no_grad():
        for batch in tqdm(data_loader,desc="Evaluating",leave=False):
            input_ids=batch["input_ids"].to(device); mask=batch["attention_mask"].to(device); labels=batch["labels"].to(device)
            logits=model(input_ids=input_ids,attention_mask=mask).logits
            losses=loss_fn(logits,labels); probs=torch.softmax(logits,dim=1); preds=torch.argmax(logits,dim=1)
            total_loss+=losses.sum().item(); total_n+=labels.size(0)
            labels_all.extend(labels.cpu().numpy()); preds_all.extend(preds.cpu().numpy()); probs_all.extend(probs.cpu().numpy()); gids_all.extend(batch["group_id"].numpy())
            for i in IDENTITY_COLUMNS: ids[i].extend(batch[i].numpy())
            if "uid" in batch: uids.extend(batch["uid"].numpy())
    labels_all=np.asarray(labels_all); preds_all=np.asarray(preds_all); probs_all=np.asarray(probs_all)
    metrics={"loss":total_loss/total_n,"accuracy":accuracy_score(labels_all,preds_all),"precision":precision_score(labels_all,preds_all,zero_division=0),"recall":recall_score(labels_all,preds_all,zero_division=0),"f1":f1_score(labels_all,preds_all,zero_division=0)}
    groups,worst=official_group_metrics(labels_all,preds_all,ids); metrics["worst_group_accuracy"]=worst
    return metrics,{"labels":labels_all,"predictions":preds_all,"probabilities":probs_all,"group_ids":np.asarray(gids_all),"identity_data":ids,"official_group_metrics":groups,"uids":np.asarray(uids) if uids else None}


# Training section

## Initialize fresh model and optimizer


In [15]:
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=2).to(device)
optimizer=AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
total_steps=len(train_loader)*NUM_EPOCHS
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=int(WARMUP_RATIO*total_steps),num_training_steps=total_steps)
loss_fn=nn.CrossEntropyLoss(reduction="none")
adv_probs=torch.ones(NUM_TRAIN_GROUPS,device=device)/NUM_TRAIN_GROUPS


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Group DRO training function


In [21]:
def train_epoch():
    global adv_probs
    model.train(); running=0; n=0; sums=torch.zeros(4,device=device); counts=torch.zeros(4,device=device)
    for batch in tqdm(train_loader,desc="Group DRO training"):
        optimizer.zero_grad(); ids=batch["input_ids"].to(device); mask=batch["attention_mask"].to(device); labels=batch["labels"].to(device); gids=batch["group_id"].to(device)
        losses=loss_fn(model(input_ids=ids,attention_mask=mask).logits,labels)
        gl=torch.zeros(4,device=device); present=torch.zeros(4,dtype=torch.bool,device=device)
        for g in range(4):
            m=gids==g
            if m.any():
                present[g]=True; gl[g]=losses[m].mean(); sums[g]+=losses[m].detach().sum(); counts[g]+=m.sum()
        with torch.no_grad():
            adv_probs[present]*=torch.exp(DRO_ETA*gl[present].detach()); adv_probs/=adv_probs.sum()
        w=adv_probs[present]; w=w/w.sum(); robust=torch.sum(w*gl[present]); robust.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step(); scheduler.step()
        running+=robust.item()*labels.size(0); n+=labels.size(0)
    return running/n,(sums/counts.clamp(min=1)).detach().cpu(),adv_probs.detach().cpu()

In [22]:
global adv_probs
history=[]; best_worst=-np.inf; best_epoch=None
for epoch in range(1,NUM_EPOCHS+1):
    print(f"EPOCH {epoch}/{NUM_EPOCHS}")
    train_loss,group_losses,weights_now=train_epoch(); val_metrics,val_outputs=evaluate_model(model,val_loader)
    print("Train robust loss:",train_loss); print("Validation:",val_metrics); print("Weights:",weights_now.numpy())
    row={"epoch":epoch,"train_robust_loss":train_loss,**{f"val_{k}":v for k,v in val_metrics.items()}}
    for g in range(4): row[f"group_{g}_loss"]=float(group_losses[g]); row[f"group_{g}_weight"]=float(weights_now[g])
    history.append(row)
    if val_metrics["worst_group_acc99999999999999999999999999uracy"]>best_worst:
        best_worst=val_metrics["worst_group_accuracy"]; best_epoch=epoch; model.save_pretrained(BEST_MODEL_DIR); tokenizer.save_pretrained(BEST_MODEL_DIR); val_outputs["official_group_metrics"].to_csv(OUTPUT_DIR/"best_validation_group_metrics.csv",index=False); print("Saved new best model.")
pd.DataFrame(history).to_csv(OUTPUT_DIR/"training_history.csv",index=False)
print("Best epoch:",best_epoch,"Best validation worst-group accuracy:",best_worst)

EPOCH 1/3


Group DRO training:   0%|          | 0/16720 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1408 [00:00<?, ?it/s]

Train robust loss: 0.40148979570235854
Validation: {'loss': 0.4364527174219068, 'accuracy': 0.8475814149665905, 'precision': 0.40819803567849267, 'recall': 0.809098132697656, 'f1': 0.5426325606181721, 'worst_group_accuracy': 0.6328125}
Weights: [3.9044227e-08 6.1567885e-01 3.3395205e-02 3.5092595e-01]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved new best model.
EPOCH 2/3


... (2 outputs left out)

Train robust loss: 0.2531572895299013
Validation: {'loss': 0.36635793023772195, 'accuracy': 0.8914023131396097, 'precision': 0.5099607182940517, 'recall': 0.7220897894318633, 'f1': 0.5977635257358987, 'worst_group_accuracy': 0.5338541666666666}
Weights: [0.28462532 0.34775347 0.23647639 0.13114484]
EPOCH 3/3


Group DRO training:   0%|          | 0/16720 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1408 [00:00<?, ?it/s]

Train robust loss: 0.16202115903481507
Validation: {'loss': 0.3823149711564145, 'accuracy': 0.9016360689946057, 'precision': 0.5463917525773195, 'recall': 0.7054032578466428, 'f1': 0.6157981444550421, 'worst_group_accuracy': 0.5338541666666666}
Weights: [0.8719214  0.02315034 0.08467948 0.02024875]
Best epoch: 1 Best validation worst-group accuracy: 0.6328125


In [23]:
if not (BEST_MODEL_DIR/"config.json").exists():
    raise FileNotFoundError("No saved best model found. Run the training section first, or restore results/group_dro/best_model/.")
best_model=AutoModelForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(device)
print("Best model loaded from:",BEST_MODEL_DIR.resolve())


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Best model loaded from: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/group_dro/best_model


## Test Evaluation


In [24]:
test_metrics,test_outputs=evaluate_model(best_model,test_loader)
print("Final Group DRO test metrics:")
for k,v in test_metrics.items(): print(f"{k}: {v:.4f}")
test_group_metrics=test_outputs["official_group_metrics"]
print("Worst evaluated test group:")
print(test_group_metrics.dropna(subset=["accuracy"]).sort_values("accuracy").iloc[0])


Evaluating:   0%|          | 0/4148 [00:00<?, ?it/s]

Final Group DRO test metrics:
loss: 0.4468
accuracy: 0.8440
precision: 0.4069
recall: 0.8117
f1: 0.5421
worst_group_accuracy: 0.6515
Worst evaluated test group:
identity          black
label                 0
label_name    non-toxic
n                  3317
accuracy       0.651492
Name: 12, dtype: object


In [25]:
prediction_df=test_dataset.to_pandas()
prediction_df["true_label"]=test_outputs["labels"]
prediction_df["prediction"]=test_outputs["predictions"]
prediction_df["prob_non_toxic"]=test_outputs["probabilities"][:,0]
prediction_df["prob_toxic"]=test_outputs["probabilities"][:,1]
prediction_df["group_id"]=test_outputs["group_ids"]
prediction_path=OUTPUT_DIR/"test_predictions.csv"
prediction_df.to_csv(prediction_path,index=False)
print("Saved:",prediction_path.resolve())
print("Shape:",prediction_df.shape)
print("Columns:",prediction_df.columns.tolist())


Saved: /content/drive/MyDrive/ele680/ele680-civilComments-20261029T191748Z-1-001/ele680-civilComments/results/group_dro/test_predictions.csv
Shape: (132730, 30)
Columns: ['uid', 'id', 'comment_text', 'toxicity', 'has_active_attrs', 'active_attrs', 'male', 'female', 'LGBTQ', 'christian', 'muslim', 'other_religions', 'black', 'white', 'identity_any', 'severe_toxicity', 'obscene', 'threat', 'insult', 'identity_attack', 'sexual_explicit', 'label', 'input_ids', 'token_type_ids', 'attention_mask', 'group_id', 'true_label', 'prediction', 'prob_non_toxic', 'prob_toxic']


In [26]:
with open(OUTPUT_DIR/"test_metrics.json","w",encoding="utf-8") as f: json.dump({k:float(v) for k,v in test_metrics.items()},f,indent=2)
test_group_metrics.to_csv(OUTPUT_DIR/"test_group_metrics.csv",index=False)
print("Saved test_metrics.json and test_group_metrics.csv")


Saved test_metrics.json and test_group_metrics.csv


In [27]:
for p in [OUTPUT_DIR/"test_predictions.csv",OUTPUT_DIR/"test_metrics.json",OUTPUT_DIR/"test_group_metrics.csv"]:
    print("FOUND" if p.exists() else "MISSING",p)


FOUND results/group_dro/test_predictions.csv
FOUND results/group_dro/test_metrics.json
FOUND results/group_dro/test_group_metrics.csv


- `best_model/`: load the trained model for later inference without retraining.
- `test_predictions.csv`: run Christian, Muslim, LGBTQ, other subgroup, comparison, and error analyses without GPU or model loading.
- `test_metrics.json`: preserve final overall performance.
- `test_group_metrics.csv`: preserve all 16 identity-label accuracies.
- `training_history.csv`: preserve epoch-level training and validation behavior.
- `experiment_config.json`: record how the experiment was produced.